In [1]:
!pip install -q transformers sentencepiece accelerate requests trafilatura newspaper4k lxml_html_clean streamlit

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.1/53.1 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 151.9/151.9 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 317.2/317.2 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.1/10.1 MB 25.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.7/80.7 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 837.9/837.9 kB 13.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 23.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 106.4/106.4 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 322.4/322.4 kB 10.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 296.7/296.7 kB 18.1 MB/s eta 0:00:00


In [2]:
import re
import requests
import torch
import trafilatura

from newspaper import Article
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

In [3]:
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Device:", device)

Device: cpu


In [4]:
MODEL_NAME = "facebook/bart-large-cnn"

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME
)

model = AutoModelForSeq2SeqLM.from_pretrained(
    MODEL_NAME
)

model = model.to(device)
model.eval()

print("BART model loaded successfully.")
print("Tokenizer vocab size:", tokenizer.vocab_size)

config.json:   0%|          | 0.00/1.58k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.63GB            

model.safetensors: downloading bytes:           |  0.00B            

[transformers] Please make sure the generation config includes `forced_bos_token_id=0`. 


Loading weights:   0%|          | 0/511 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/363 [00:00<?, ?B/s]

BART model loaded successfully.
Tokenizer vocab size: 50265


In [5]:
def create_chunks(
    article,
    chunk_size=900,
    stride=100
):
    if stride >= chunk_size:
        raise ValueError(
            "Stride must be smaller than chunk size."
        )

    tokens = tokenizer(
        article,
        add_special_tokens=False,
        truncation=False
    )["input_ids"]

    chunks = []

    start = 0
    step = chunk_size - stride

    while start < len(tokens):

        end = start + chunk_size

        chunk = tokens[start:end]

        chunks.append(chunk)

        if end >= len(tokens):
            break

        start += step

    return chunks

In [6]:
def direct_bart_summarize(
    article,
    max_input_length=1024,
    max_summary_length=120,
    min_summary_length=30
):

    inputs = tokenizer(
        article,
        return_tensors="pt",
        max_length=max_input_length,
        truncation=True
    )

    inputs = {
        key: value.to(device)
        for key, value in inputs.items()
    }

    with torch.no_grad():

        summary_ids = model.generate(
            **inputs,
            max_length=max_summary_length,
            min_length=min_summary_length,
            num_beams=4,
            length_penalty=2.0,
            early_stopping=True
        )

    summary = tokenizer.decode(
        summary_ids[0],
        skip_special_tokens=True,
        clean_up_tokenization_spaces=True
    )

    return summary

In [7]:
def summarize_chunk(
    chunk_text,
    max_input_length=1024,
    max_summary_length=150,
    min_summary_length=40
):

    inputs = tokenizer(
        chunk_text,
        return_tensors="pt",
        max_length=max_input_length,
        truncation=True
    )

    inputs = {
        key: value.to(device)
        for key, value in inputs.items()
    }

    with torch.no_grad():

        summary_ids = model.generate(
            **inputs,
            max_length=max_summary_length,
            min_length=min_summary_length,
            num_beams=4,
            length_penalty=2.0,
            early_stopping=True
        )

    summary = tokenizer.decode(
        summary_ids[0],
        skip_special_tokens=True,
        clean_up_tokenization_spaces=True
    )

    return summary

In [8]:
def hierarchical_summarize(article):

    chunks = create_chunks(article)

    print(f"Created {len(chunks)} chunks")

    chunk_summaries = []

    for i, chunk in enumerate(chunks):

        print(
            f"Summarizing chunk {i + 1}/{len(chunks)}"
        )

        chunk_text = tokenizer.decode(
            chunk,
            skip_special_tokens=True,
            clean_up_tokenization_spaces=True
        )

        summary = summarize_chunk(chunk_text)

        chunk_summaries.append(summary)

    combined_summary = " ".join(
        chunk_summaries
    )

    print("Generating final summary...")

    final_summary = direct_bart_summarize(
        combined_summary,
        max_input_length=1024,
        max_summary_length=120,
        min_summary_length=30
    )

    return final_summary

In [9]:
def summarize_news(article):

    tokens = tokenizer(
        article,
        add_special_tokens=False,
        truncation=False
    )["input_ids"]

    token_count = len(tokens)

    print("Input tokens:", token_count)

    if token_count <= 1024:

        print("Using Direct BART")

        summary = direct_bart_summarize(
            article
        )

        method = "Direct BART"

    else:

        print("Using Hierarchical BART")

        summary = hierarchical_summarize(
            article
        )

        method = "Hierarchical BART"

    return {
        "summary": summary,
        "method": method,
        "input_tokens": token_count
    }

In [10]:
from google.colab import userdata

NEWS_API_KEY = userdata.get(
    "NEWS_API_KEY"
)

print(
    "API key loaded:",
    NEWS_API_KEY is not None
)

API key loaded: True


In [11]:
def fetch_news(
    query,
    page_size=5
):

    url = "https://newsapi.org/v2/everything"

    params = {
        "q": query,
        "language": "en",
        "sortBy": "publishedAt",
        "pageSize": page_size
    }

    headers = {
        "X-Api-Key": NEWS_API_KEY
    }

    response = requests.get(
        url,
        params=params,
        headers=headers,
        timeout=15
    )

    response.raise_for_status()

    data = response.json()

    if data.get("status") != "ok":

        raise Exception(
            data.get(
                "message",
                "NewsAPI request failed"
            )
        )

    return data.get(
        "articles",
        []
    )

In [12]:
def clean_article_text(text):

    if not text:
        return None

    text = re.sub(
        r"\s+",
        " ",
        text
    ).strip()

    boilerplate_patterns = [

        r"Add this site to your preferred sources.*$",

        r"You may also like.*$",

        r"Read more.*$",

        r"Subscribe to.*$",

        r"Follow us on.*$"
    ]

    for pattern in boilerplate_patterns:

        text = re.sub(
            pattern,
            "",
            text,
            flags=re.IGNORECASE
        ).strip()

    return text

In [13]:
def validate_article(
    text,
    min_chars=500
):

    if not text:
        return False

    text = text.strip()

    if len(text) < min_chars:
        return False

    if len(text.split()) < 100:
        return False

    return True

In [14]:
def extract_article_text(url):

    # -------------------------
    # Method 1: Trafilatura
    # -------------------------

    try:

        downloaded = trafilatura.fetch_url(
            url
        )

        if downloaded:

            text = trafilatura.extract(
                downloaded,
                include_comments=False,
                include_tables=False
            )

            text = clean_article_text(
                text
            )

            if validate_article(text):

                return text

    except Exception as e:

        print(
            "Trafilatura failed:",
            e
        )


    # -------------------------
    # Method 2: Newspaper4k
    # -------------------------

    try:

        article = Article(url)

        article.download()

        article.parse()

        text = article.text

        text = clean_article_text(
            text
        )

        if validate_article(text):

            return text

    except Exception as e:

        print(
            "Newspaper4k failed:",
            e
        )


    return None

In [15]:
def fetch_and_summarize(
    query,
    page_size=5
):

    articles = fetch_news(
        query=query,
        page_size=page_size
    )

    results = []

    seen_urls = set()

    for i, article in enumerate(
        articles
    ):

        print(
            f"\nProcessing article "
            f"{i + 1}/{len(articles)}"
        )

        title = article.get(
            "title"
        )

        url = article.get(
            "url"
        )

        if not url:

            print(
                "Skipped: no URL"
            )

            continue

        if url in seen_urls:

            print(
                "Skipped: duplicate URL"
            )

            continue

        seen_urls.add(url)

        print(
            "Title:",
            title
        )

        # Extract article

        text = extract_article_text(
            url
        )

        if not text:

            print(
                "Skipped: article extraction failed"
            )

            continue

        print(
            "Extracted characters:",
            len(text)
        )

        # Summarize

        try:

            result = summarize_news(
                text
            )

            results.append({

                "title": title,

                "source": article.get(
                    "source",
                    {}
                ).get(
                    "name"
                ),

                "publishedAt":
                    article.get(
                        "publishedAt"
                    ),

                "url": url,

                "summary":
                    result["summary"],

                "method":
                    result["method"],

                "input_tokens":
                    result["input_tokens"]

            })

            print(
                "Summary generated successfully"
            )

        except Exception as e:

            print(
                "Summarization failed:",
                e
            )

    return results

In [16]:
query = input(
    "Enter news topic: "
)

Enter news topic: india


In [17]:
results = fetch_and_summarize(
    query=query,
    page_size=5
)

print(
    f"\nSuccessfully processed "
    f"{len(results)} articles."
)


Processing article 1/5
Title: L&T sees $150-billion global modularisation opportunity
Extracted characters: 3589
Input tokens: 679
Using Direct BART


[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer RobertaTokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.
ERROR:trafilatura.downloads:not a 200 response: 403 for URL https://www.business-standard.com/external-affairs-defence-security/news/india-flags-red-sea-concerns-reiterates-impact-on-india-us-trade-ties-126091801326_1.html


Summary generated successfully

Processing article 2/5
Title: India flags Red Sea concerns, reiterates impact on India-US trade ties
Extracted characters: 3201
Input tokens: 590
Using Direct BART
Summary generated successfully

Processing article 3/5
Title: AAHRPP Accredits Second Organization in Mexico: Instituto Nacional de Ciencias Médicas y Nutrición Salvador Zubirán
Extracted characters: 2142
Input tokens: 452
Using Direct BART
Summary generated successfully

Processing article 4/5
Title: Listing of Tata Sons shares an imperative: Shapoorji Pallonji group
Extracted characters: 2766
Input tokens: 539
Using Direct BART
Summary generated successfully

Processing article 5/5
Title: Rebuilding damaged oil & gas infrastructure in West Asia offers EIL over $1 billion opportunity
Extracted characters: 3347
Input tokens: 674
Using Direct BART
Summary generated successfully

Successfully processed 5 articles.


In [18]:
results = fetch_and_summarize(
    query=query,
    page_size=5
)

print(
    f"\nSuccessfully processed "
    f"{len(results)} articles."
)


Processing article 1/5
Title: L&T sees $150-billion global modularisation opportunity
Extracted characters: 3589
Input tokens: 679
Using Direct BART


ERROR:trafilatura.downloads:not a 200 response: 403 for URL https://www.business-standard.com/external-affairs-defence-security/news/india-flags-red-sea-concerns-reiterates-impact-on-india-us-trade-ties-126091801326_1.html


Summary generated successfully

Processing article 2/5
Title: India flags Red Sea concerns, reiterates impact on India-US trade ties
Extracted characters: 3201
Input tokens: 590
Using Direct BART
Summary generated successfully

Processing article 3/5
Title: AAHRPP Accredits Second Organization in Mexico: Instituto Nacional de Ciencias Médicas y Nutrición Salvador Zubirán
Extracted characters: 2142
Input tokens: 452
Using Direct BART
Summary generated successfully

Processing article 4/5
Title: Listing of Tata Sons shares an imperative: Shapoorji Pallonji group
Extracted characters: 2766
Input tokens: 539
Using Direct BART
Summary generated successfully

Processing article 5/5
Title: Rebuilding damaged oil & gas infrastructure in West Asia offers EIL over $1 billion opportunity
Extracted characters: 3347
Input tokens: 674
Using Direct BART
Summary generated successfully

Successfully processed 5 articles.


In [19]:
for i, result in enumerate(results):

    print("\n")
    print("=" * 80)

    print(
        f"ARTICLE {i + 1}"
    )

    print(
        "Title:",
        result["title"]
    )

    print(
        "Source:",
        result["source"]
    )

    print(
        "Published:",
        result["publishedAt"]
    )

    print(
        "Method:",
        result["method"]
    )

    print(
        "Input tokens:",
        result["input_tokens"]
    )

    print("\nSUMMARY:")

    print(
        result["summary"]
    )

    print("\nURL:")

    print(
        result["url"]
    )



ARTICLE 1
Title: L&T sees $150-billion global modularisation opportunity
Source: BusinessLine
Published: 2026-09-18T15:36:07Z
Method: Direct BART
Input tokens: 679

SUMMARY:
Larsen & Toubro (L&T) expects the $150 billion global market for modularisation of industrial projects to offer opportunities worth about $30 billion over the next five years. Rising labour costs and shortage of skilled workers encourage project developers to shift from conventional site construction to factory-built modules.

URL:
https://www.thehindubusinessline.com/companies/lt-sees-150-billion-global-modularisation-opportunity/article71481413.ece


ARTICLE 2
Title: India flags Red Sea concerns, reiterates impact on India-US trade ties
Source: Business Standard
Published: 2026-09-18T15:35:19Z
Method: Direct BART
Input tokens: 590

SUMMARY:
MEA says attacks threatening navigation in Bab-el-Mandeb are unacceptable. While reiterating concerns over the impact of proposed US sanctions legislation on trade ties.

UR

In [20]:
if len(results) > 0:

    test_article = results[0]

    print(
        "TITLE:",
        test_article["title"]
    )

    print("\nSUMMARY:")

    print(
        test_article["summary"]
    )

TITLE: L&T sees $150-billion global modularisation opportunity

SUMMARY:
Larsen & Toubro (L&T) expects the $150 billion global market for modularisation of industrial projects to offer opportunities worth about $30 billion over the next five years. Rising labour costs and shortage of skilled workers encourage project developers to shift from conventional site construction to factory-built modules.
